# 🔁 Planning: Decompose → Execute → Replan

## Learning Objectives
In this notebook, you will learn:
1. **Plans as living documents** - Why a fixed plan silently mishandles a bad tool result, and what replanning adds
2. **Per-step evaluation** - Judge every step's result with a structured check: *did this actually answer the step?*
3. **The replanner** - Rewrite the *remaining* plan from the history and the reason the last step failed
4. **Routing and a stopping rule** - Pure router functions plus a max-iteration guard
5. **Proving both paths** - A clean run with no replan, a forced wrong-entity result that triggers one, and a dead tool that hits the cap

## Where this fits
`01_planning_react_vs_plan.ipynb` built a planning agent that generates a fixed, ordered list of steps up front and then marches through them one by one, trusting that every step will succeed exactly as written. That is a great fit when the world is predictable - but real tool calls fail, return ambiguous data, or turn out to answer the wrong question. This notebook lets the agent **notice** when a step's result doesn't satisfy what was asked, and **replan** instead of blindly continuing. `03_planning_agent_deep_research.ipynb` then scales planning up to a whole research report.

## Prerequisites
- `01_planning_react_vs_plan.ipynb` (the fixed plan-and-execute agent this notebook extends)
- Per-criterion structured judgements - `modules/09_evaluator_optimizer/notebooks/01_evaluator_optimizer_langgraph.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` and `TAVILY_API_KEY` in the project-root `.env`

---
## 📖 Part 1: Plans as Living Documents

**Plan → Execute → Replan** is a Planning architecture where the plan is treated as a *living document* rather than a fixed script. A Planner decomposes the goal into steps, an Executor carries out one step at a time, but after every step an explicit evaluation decides whether that step's result is actually good enough. When it isn't, a Replanner is invoked: it looks at everything learned so far (including *why* the last step failed) and produces a fresh list of remaining steps — it does not simply retry the same query verbatim.

### High-level Workflow

1. **Plan:** Decompose the goal into an ordered list of steps (structured output).
2. **Execute:** Run the next step (a tool call) and capture its result.
3. **Evaluate:** Check whether the result actually satisfies that step — not just "did the tool run without error", but "does this answer what the step asked for?"
4. **Route:**
   - Satisfied + steps remain → go back to **Execute**.
   - Satisfied + no steps remain → go to **Synthesize**.
   - Not satisfied → go to **Replan**.
5. **Replan:** Given the goal, the history of completed/failed steps, and the reason the last step fell short, generate a *new* list of remaining steps. Loop back to **Execute**.
6. A **max-iteration guard** forces the loop to exit to **Synthesize** even if the plan never fully succeeds, so the agent can still give a best-effort answer instead of looping forever.
7. **Synthesize:** Combine everything learned into a final answer.

| Property | Value |
|---|---|
| Origin | Long lineage in classical AI (STRIPS, HTN); modern LangGraph idiom: [plan-and-execute tutorial](https://langchain-ai.github.io/langgraph/tutorials/plan-and-execute/plan-and-execute/). The replan step specifically echoes Plan-and-Solve + verifier composition, canonically Hu et al., *Tree-Planner* (2023) ([arXiv:2305.10142](https://arxiv.org/abs/2305.10142)). |

### When to Use / Applications
* **Research agents over live data:** web search results are noisy and sometimes off-topic; a fixed plan has no way to recover from a bad search.
* **Multi-hop question answering:** where an early ambiguous sub-question (e.g. a name or term with two meanings) can silently poison every step that follows it.
* **Long-horizon workflows in changing environments:** e.g. a step that depends on an API response whose shape or availability isn't fully known in advance.

### Strengths & Weaknesses
* **Strengths:**
  * **Adaptive:** Recovers from bad tool results instead of propagating them downstream.
  * **Still structured:** Unlike ReAct, replanning still produces an explicit, inspectable plan at every stage — you can always print "what does the agent currently believe the remaining steps are?"
  * **Learns from failure, not just repeats it:** the Replanner sees *why* a step failed and can change strategy (e.g. use a more specific search query), rather than mechanically retrying.
* **Weaknesses:**
  * **More LLM calls:** every step now costs a planning call *and* an evaluation call, and a possible replanning call — more expensive than the fixed-plan version.
  * **Evaluator quality is a bottleneck:** if the "is this satisfied?" check is too lenient, bad data slips through; if too strict, the agent replans needlessly and can loop.
  * **Needs a stopping condition:** without a max-iteration guard, a persistently ambiguous goal could cause the agent to replan forever.

---
## 📦 Step 1: Environment Setup

Load the API keys and create the chat model through the course factory.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY and TAVILY_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: Planner, step evaluator, replanner and synthesizer
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🧱 Part 2: Structured Schemas, Shared State and the Tool

Three small structures keep every LLM call in this graph machine-checkable:
1. `Plan` - an ordered list of step descriptions (used by both the Planner and the Replanner).
2. `StepEvaluation` - the evaluator's judgment of whether a step's result actually satisfies that step.
3. `PlanExecuteState` - the LangGraph state that threads through every node: the goal, the current remaining plan, the history of completed steps (with their evaluations), an iteration counter with a max guard, and the eventual final answer.

The tool is the same Tavily `web_search` as in `01_planning_react_vs_plan.ipynb`, so the two notebooks stay comparable.

In [ ]:
# ============================================================================
# SCHEMAS & STATE
# ============================================================================
from typing import List, Optional, TypedDict

from pydantic import BaseModel, Field


class Plan(BaseModel):
    """An ordered list of steps still required to satisfy the goal."""

    steps: List[str] = Field(
        description=(
            "Ordered list of fact-finding steps. Each step is one specific, unambiguous "
            "question that a single web search can answer."
        )
    )


class StepEvaluation(BaseModel):
    """Judges whether a tool result actually satisfies the step it was meant to answer."""

    satisfied: bool = Field(
        description="True if the search result states the fact the step asks for, about the right entity."
    )
    reason: str = Field(
        description="One sentence explaining why the result does, or does not, satisfy the step."
    )


class PlanExecuteState(TypedDict):
    goal: str
    plan: List[str]
    completed_steps: List[dict]  # {"step", "result", "satisfied", "reason"}
    iterations: int
    max_iterations: int
    final_answer: Optional[str]


print(f"✅ PlanExecuteState fields: {list(PlanExecuteState.__annotations__)}")

In [ ]:
# ============================================================================
# TOOL: Tavily web search
# ============================================================================
from langchain_core.tools import tool
from langchain_tavily import TavilySearch

tavily_search_tool = TavilySearch(max_results=3)


@tool
def web_search(query: str) -> str:
    """Search the web for the given query and return the results as a string."""
    print(f"   🔎 TOOL: searching for '{query}'")
    return str(tavily_search_tool.invoke(query))


print(f"✅ Tool ready: {web_search.name}")

---
## ⚙️ Part 3: Planner, Executor (with Evaluation), Replanner and Synthesizer

1. **`planner_node`** - decomposes the goal into an initial ordered plan of *lookups*. Comparisons and arithmetic are left to the synthesizer, so every step is something a search can answer.
2. **`executor_node`** - runs the *next* step's tool call, then immediately asks the LLM to judge whether the result actually satisfies that step (a lightweight LLM-as-judge check, per step). The executor is built by a function that takes the search tool as an argument, so Part 5 can swap in misbehaving tools without touching anything else.
3. **`replanner_node`** - only invoked when a step's evaluation comes back unsatisfied. It is shown the goal, the full history of what has and hasn't worked, and the reason the last step failed, and produces a **new** list of remaining steps - it is not a retry of the same query, it is a fresh planning pass informed by failure.
4. **`synthesizer_node`** - once the plan is empty (or the iteration guard trips), combines the *accepted* results into a final answer, being explicit about any fact that was never found.

In [ ]:
# ============================================================================
# PLANNER NODE
# ============================================================================
planner_llm = llm.with_structured_output(Plan)


def planner_node(state: PlanExecuteState):
    print("📋 PLANNER: decomposing goal into an initial plan...")
    prompt = f"""You are an expert planner. Break the user's goal down into an ordered list of
fact-finding steps. Each step must be one specific question that a single web search can answer.
Only plan the lookups: comparisons, calculations and the final answer are done later by a
synthesizer, so do not add steps for them.
Avoid phrasing that could be read more than one way (for example, prefer "population of the
country of Georgia (Caucasus region)" over the ambiguous "population of Georgia").

Goal: {state['goal']}
"""
    result = planner_llm.invoke(prompt)
    print(f"📋 PLANNER: initial plan: {result.steps}")
    return {"plan": result.steps, "completed_steps": [], "iterations": 0}

In [ ]:
# ============================================================================
# EXECUTOR NODE (execute + evaluate), built around a swappable search tool
# ============================================================================
evaluator_llm = llm.with_structured_output(StepEvaluation)

EVAL_PROMPT = """A plan step and a search result are given below. Decide whether the result states the
fact the step asks for.
- Mark it NOT satisfied if the result is about the wrong entity (a different country, person or
  thing with a similar name), or does not contain the requested fact at all.
- A recent estimate or official figure counts as satisfying a "current" question; the result
  does not have to be perfect, only to answer the step.

Step: {step}
Search result: {result}
"""


def make_executor_node(search):
    """Build the executor around any tool-like object with .invoke(query) -> str."""

    def executor_node(state: PlanExecuteState):
        step = state["plan"][0]
        print(f"⚙️  EXECUTOR: running step -> '{step}'")
        result = search.invoke(step)

        evaluation = evaluator_llm.invoke(EVAL_PROMPT.format(step=step, result=result))
        print(f"🧑‍⚖️ EVALUATOR: satisfied={evaluation.satisfied} | {evaluation.reason}")

        completed_steps = state["completed_steps"] + [
            {"step": step, "result": result, "satisfied": evaluation.satisfied, "reason": evaluation.reason}
        ]
        # The step is always popped: whether or not it succeeded, the Replanner (not a raw
        # retry) is responsible for deciding what should happen next.
        return {
            "plan": state["plan"][1:],
            "completed_steps": completed_steps,
            "iterations": state["iterations"] + 1,
        }

    return executor_node


print("✅ Executor factory defined")

In [ ]:
# ============================================================================
# REPLANNER NODE
# ============================================================================
replanner_llm = llm.with_structured_output(Plan)


def replanner_node(state: PlanExecuteState):
    print("🔁 REPLANNER: last step was insufficient - regenerating the remaining plan...")

    history = "\n".join(
        f"- [{'OK' if c['satisfied'] else 'INSUFFICIENT'}] Step: {c['step']}\n  Reason: {c['reason']}"
        for c in state["completed_steps"]
    )
    prompt = f"""You are replanning a multi-step research task because the most recent step
did not produce a satisfactory result.

Overall goal: {state['goal']}

History of steps attempted so far (do not repeat steps already marked OK):
{history}

Previously remaining (now stale) steps, for reference only: {state['plan']}

Using what you have learned from the INSUFFICIENT step above, produce a NEW ordered list of
the lookups still required to fully satisfy the goal:
- Do not resubmit the failed step unchanged: rephrase it using the reason it failed (for
  example add the disambiguating detail the result was missing), or split it into smaller lookups.
- Only plan lookups. Comparisons, calculations and the final answer are done later by a
  synthesizer, so do not add steps for them.
"""
    new_plan = replanner_llm.invoke(prompt)
    print(f"🔁 REPLANNER: new remaining plan: {new_plan.steps}")
    return {"plan": new_plan.steps}

In [ ]:
# ============================================================================
# SYNTHESIZER NODE
# ============================================================================
def synthesizer_node(state: PlanExecuteState):
    print("🧩 SYNTHESIZER: generating final answer...")
    # Only accepted results are passed on as data: a rejected result must not leak into the answer.
    accepted = [c for c in state["completed_steps"] if c["satisfied"]]
    rejected = [c["step"] for c in state["completed_steps"] if not c["satisfied"]]
    data = "\n\n".join(f"Step: {c['step']}\nResult: {c['result']}" for c in accepted) or "(none)"
    prompt = f"""You are an expert synthesizer. Answer the user's original goal using ONLY the
accepted search results below - not your own knowledge. If a fact the goal needs is not in
the accepted results, say plainly that it could not be found rather than filling it in.

Goal: {state['goal']}

Accepted search results:
{data}

Lookups whose results were rejected (for context only; a later accepted step may cover them):
{rejected or "(none)"}
"""
    return {"final_answer": llm.invoke(prompt).content}


print("✅ Planner, Executor (with evaluation), Replanner, and Synthesizer nodes defined")

---
## 🚦 Part 4: Routing and a Stopping Rule

* After **`execute`**: if the max-iteration guard has tripped, go straight to `synthesize`. Otherwise, if the last step was unsatisfied, go to `replan`; if satisfied and the plan is empty, go to `synthesize`; if satisfied and steps remain, loop back to `execute`.
* After **`replan`**: go back to `execute` with the freshly regenerated plan (or straight to `synthesize` in the edge case where the Replanner determines nothing more is needed).

This is the key structural difference from `01_planning_react_vs_plan.ipynb`: that notebook's router only ever asks "is the plan empty yet?". Here the router also asks "did the step we just ran actually succeed?", and a failed step is routed to a dedicated `replan` node instead of silently being treated as done. Both routers are pure functions of state - the LLM verdict was already written to state by the executor.

In [ ]:
# ============================================================================
# ROUTERS: Pure functions of state
# ============================================================================
def route_after_execute(state: PlanExecuteState):
    if state["iterations"] >= state["max_iterations"]:
        print("🛑 ROUTER: max iterations reached - forcing synthesis")
        return "synthesize"
    if not state["completed_steps"][-1]["satisfied"]:
        return "replan"
    return "execute" if state["plan"] else "synthesize"


def route_after_replan(state: PlanExecuteState):
    return "execute" if state["plan"] else "synthesize"


print("✅ Routers defined")

In [ ]:
# ============================================================================
# GRAPH ASSEMBLY: build_plan_execute_app(search) wires the four nodes
# ============================================================================
from langgraph.graph import END, START, StateGraph


def build_plan_execute_app(search):
    graph_builder = StateGraph(PlanExecuteState)
    graph_builder.add_node("plan", planner_node)
    graph_builder.add_node("execute", make_executor_node(search))
    graph_builder.add_node("replan", replanner_node)
    graph_builder.add_node("synthesize", synthesizer_node)

    graph_builder.add_edge(START, "plan")
    graph_builder.add_edge("plan", "execute")
    graph_builder.add_conditional_edges(
        "execute", route_after_execute, {"execute": "execute", "replan": "replan", "synthesize": "synthesize"}
    )
    graph_builder.add_conditional_edges("replan", route_after_replan, {"execute": "execute", "synthesize": "synthesize"})
    graph_builder.add_edge("synthesize", END)
    return graph_builder.compile()


plan_execute_app = build_plan_execute_app(web_search)
print("✅ Plan -> Execute -> Replan agent compiled")

In [ ]:
# ============================================================================
# INSPECT THE GRAPH: Print nodes and edges (no network call needed)
# ============================================================================
g = plan_execute_app.get_graph()

print("🧩 Nodes:", [n for n in g.nodes if not n.startswith("__")])
print("🔗 Edges:")
for e in g.edges:
    kind = "conditional" if e.conditional else "fixed"
    print(f"   {e.source:>18} ──▶ {e.target:<18} ({kind})")

In [ ]:
# ============================================================================
# RUN HELPER: Start a goal and print the step-by-step trace
# ============================================================================
from IPython.display import Markdown, display


def run_goal(app, goal: str, max_iterations: int = 6) -> dict:
    final_state = app.invoke(
        {"goal": goal, "plan": [], "completed_steps": [], "iterations": 0,
         "max_iterations": max_iterations, "final_answer": None}
    )
    replans = sum(1 for c in final_state["completed_steps"] if not c["satisfied"])
    print(f"\n📊 {final_state['iterations']} steps executed, {replans} unsatisfied (each triggers a replan unless the cap is hit)")
    for i, c in enumerate(final_state["completed_steps"], start=1):
        print(f"   {i}. [{'OK' if c['satisfied'] else 'INSUFFICIENT'}] {c['step']}")
    display(Markdown(final_state["final_answer"]))
    return final_state


print("✅ run_goal helper ready")

---
## 🧪 Part 5: Proving Both Paths

A replanning graph has three behaviours worth seeing: the happy path (no replan), a recovery (one replan), and a give-up (the cap). Real web search rarely fails on demand, so the last two runs swap in **stub search tools** that fail in a known way. Because the executor takes the tool as an argument, nothing else in the graph changes.

### 5.1 ✅ A Straightforward Run (No Replanning Needed)

A goal whose steps are each unambiguous, with real search. Every step should be satisfied on the first try.

In [ ]:
# ============================================================================
# RUN 1: STRAIGHTFORWARD GOAL, REAL SEARCH
# ============================================================================
simple_goal = (
    "Find the current capital city of Japan and the current capital city of Canada, "
    "and state which country has the larger total population."
)

final_state_simple = run_goal(plan_execute_app, simple_goal)

On a well-posed goal like this one, the graph behaves exactly like the fixed-plan agent from `01_planning_react_vs_plan.ipynb`: `plan → execute → … → synthesize`, with every `StepEvaluation.satisfied` coming back `True` and the `replan` node never firing. The extra machinery is there, but it stays invisible when it isn't needed - replanning should be a safety net, not overhead that changes behavior on easy tasks. (It still costs one evaluation call per step.)

### 5.2 🔁 A Wrong-Entity Result (Forces a Replan)

**"Georgia"** refers both to a country in the Caucasus (capital Tbilisi) and to a U.S. state. Search engines often resolve a query about the country to the far more searched-for U.S. state. To make that failure reproducible, `misleading_search` answers its **first** query with a U.S.-state snippet, whatever was asked, and uses real search after that.

The Executor's evaluator should catch the mismatch and mark the step unsatisfied - which routes to the Replanner. The Replanner then has to look at *why* the step failed and produce a sharper follow-up rather than simply resubmitting the same query.

In [ ]:
# ============================================================================
# STUB TOOL: The first search returns the wrong Georgia; later searches are real
# ============================================================================
US_STATE_SNIPPET = (
    "Georgia (U.S. state) - Wikipedia: Georgia is a state in the Southeastern United States. "
    "With a population of about 11 million (2023 estimate), it is the eighth most populous U.S. state. "
    "Its capital and largest city is Atlanta."
)


class MisleadingSearch:
    """Returns a wrong-entity result for the first query, then delegates to real search."""

    def __init__(self):
        self.calls = 0

    def invoke(self, query: str) -> str:
        self.calls += 1
        if self.calls == 1:
            print(f"   🎭 STUB: returning a U.S.-state result for '{query}'")
            return US_STATE_SNIPPET
        return web_search.invoke(query)


tricky_goal = (
    "Find the population of the country of Georgia (the one in the Caucasus region, "
    "capital Tbilisi, formerly part of the Soviet Union - NOT the U.S. state), "
    "and compare it to the population of the U.S. state of Georgia. Say which is larger."
)

final_state_tricky = run_goal(build_plan_execute_app(MisleadingSearch()), tricky_goal)

Read the trace: the first step comes back `INSUFFICIENT` with a reason naming the wrong entity, the `🔁 REPLANNER` line shows a new remaining plan, and the next attempt at the country's population is `OK`. The fixed-plan agent in `01_planning_react_vs_plan.ipynb` cannot produce this: given the same first result, it would have popped that step off the list and moved on, silently baking the U.S. state's 11 million into the country's row of the final answer.

Replanning is not retrying. Compare the failed step with the replanner's first new step: it is told not to resubmit a failed step unchanged, so it rewrites the query using the evaluator's `reason`. When the original query was already specific, the rewrite can be small - what matters is that the new result is evaluated again before the agent moves on. Notice also that the synthesizer never sees the rejected U.S.-state snippet: only accepted results are passed on as data.

### 5.3 🛑 A Dead Tool (Hits the Cap)

If a goal can never be satisfied - here, a search tool that always returns nothing - the evaluator keeps rejecting, the replanner keeps producing new plans, and only the `max_iterations` guard ends the loop. The synthesizer then receives no accepted results at all and must say so instead of filling the gaps from the model's own knowledge - which is why its prompt says "ONLY the collected data".

In [ ]:
# ============================================================================
# STUB TOOL: A search that never finds anything - the cap must end the loop
# ============================================================================
class DeadSearch:
    def invoke(self, query: str) -> str:
        print(f"   💀 STUB: no results for '{query}'")
        return "No results found."


final_state_dead = run_goal(build_plan_execute_app(DeadSearch()), simple_goal, max_iterations=3)
assert final_state_dead["iterations"] == 3, "the cap should stop the loop after exactly 3 executed steps"
print("✅ Loop stopped at the cap")

---
## 📝 Summary

In this notebook, we learned:

### 1. Plans as living documents
- A plan is more useful as a *living* artifact than a fixed script: pairing every executed step with an explicit evaluation is what makes adaptive behavior possible.

### 2. Per-step evaluation
- The evaluator answers one structured question per step - *does this result state the requested fact, about the right entity?* - and writes the verdict to state.
- Evaluator quality is the bottleneck: too strict and it replans on good data, too lenient and wrong data slips through. Spell out what counts as "satisfied".

### 3. The replanner
- Replanning is not retrying. The `replanner_node` sees the goal, the complete history (including *why* the last attempt failed), and produces a new remaining plan - often a more specific query rather than the same call.

### 4. Routing and a stopping rule
- Pure routers read the verdict from state. A max-iteration guard is essential once the plan can change size at runtime; without one, an unsatisfiable goal has no natural exit.
- This costs more than the fixed-plan version - every step carries an evaluation call, and a failed step adds a replanning call.

### 5. Proving both paths
- Building the executor around a swappable tool lets stub tools reproduce a wrong-entity result (one replan, then recovery) and a dead tool (cap, then an honest "unresolved" answer) on demand.

### Next Steps
- **`03_planning_agent_deep_research.ipynb`**: a planner that designs a whole research report and dispatches nested section-writing workflows in parallel